# 민원 분류 백엔드 — Colab 실행 (L4 권장, T4 가능)

`google/gemma-4-E4B-it`(⑤, Ollama 의 `gemma4:e4b` 와 같은 모델)와 `BAAI/bge-m3`(③)를 GPU에 올려 파이프라인 ②~⑤를 돌립니다.

**위에서부터 셀을 차례로 실행하세요.** 3번 셀 다음에 런타임 재시작이 한 번 있습니다.

| | 어디서 | 무엇을 |
|---|---|---|
| ①② 추출 | 윈도우 (PaddleOCR) | 파일 → 텍스트 |
| ③④⑤ 판정 | **여기 Colab (L4 권장)** | 텍스트 → 의도·카테고리·도구 호출 JSON |

> Colab에는 PaddleOCR을 설치하지 않습니다. 이미지·스캔 PDF 업로드는 503이 뜨는 게 정상이고,
> `/analyze/text`로 문장을 넣어 테스트하시면 됩니다.


## 0. GPU 확인

먼저 **[런타임] → [런타임 유형 변경] → 하드웨어 가속기 = `L4 GPU`**(권장, bfloat16 지원) 또는 `T4 GPU` 로 바꾸고 저장하세요.
Gemma 4 E4B 는 4bit 로도 약 9~10GB 라 T4(16GB)에도 들어가지만, T4 는 float16 이라 Gemma 에서 값이 넘칠 수 있습니다.

아래에서 `NVIDIA L4`(또는 `Tesla T4`)가 보여야 합니다. 안 보이면 런타임 유형이 CPU입니다.


In [ ]:
!nvidia-smi


## 1. 코드 올리기 (구글 드라이브)

`MyDrive/backend` 폴더에 백엔드 코드를 올려 두고 씁니다.
런타임이 끊겨도 코드와 데이터(`data/`)가 드라이브에 남습니다.

처음이면 PC 에서 `backend` 폴더를 통째로 구글 드라이브 `내 드라이브` 바로 아래에 올리세요.
아래 셀을 실행하면 드라이브 접근 허용 창이 뜹니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/backend
!ls

## 2. 설치

Colab에는 torch가 이미 CUDA판으로 깔려 있으므로 **덮어쓰지 않도록 `torch` 줄만 빼고** 설치합니다.

5~10분 걸립니다.


In [ ]:
# 웹·추출 (torch 없음)
!pip install -q -r requirements.txt

# 모델 스택 — Colab의 CUDA torch를 건드리지 않도록 torch 줄 제외
!grep -v '^torch' requirements-model.txt > /tmp/req_model.txt
!pip install -q -r /tmp/req_model.txt

# 4bit 양자화 — Gemma 4 E4B 를 올리려면 사실상 필수
!pip install -q bitsandbytes

# Gemma 4 는 transformers 5.5 이상이 필요합니다
!pip install -q --upgrade 'transformers>=5.5.0'

print('설치 완료 — 아래 셀을 읽고 런타임을 재시작하세요')


## 3. 런타임 재시작 ⚠️

새로 설치한 transformers를 적용하려면 **한 번 재시작해야 합니다.**

아래 셀을 실행하면 세션이 끊깁니다. 끊긴 뒤 **4번 셀부터** 이어서 실행하세요.
(1번 셀의 `%cd`는 재시작하면 풀리므로 4번 셀에서 다시 잡아줍니다)


In [ ]:
import os
os.kill(os.getpid(), 9)   # 런타임 재시작. '세션이 다시 시작되었습니다' 는 정상입니다


## 4. 재시작 후 — 경로 다시 잡고 환경 확인

L4·A100 이면 `bf16 지원: True`, T4 면 `False` 입니다. T4는 Turing이라 bfloat16이 없어 코드가 float16을 씁니다. (Gemma 는 bfloat16 GPU 권장)


In [ ]:
# 재시작하면 작업 경로가 풀리므로 드라이브의 backend 로 다시 이동합니다
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/backend

import torch, transformers
print('torch          :', torch.__version__)
print('CUDA 사용 가능  :', torch.cuda.is_available())
print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')
print('bf16 지원      :', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else '-', ' <- L4·A100 True / T4 False')
print('transformers   :', transformers.__version__)

import bitsandbytes
print('bitsandbytes   :', bitsandbytes.__version__)


## 5. `.env` 만들기

권장값입니다. 세 가지가 핵심입니다.

- `LLM_LOAD_4BIT=true` — 끄면 16bit 로 약 16GB 라 들어가지 않습니다 (4bit 여도 약 9~10GB)
- `LLM_4BIT_COMPUTE_DTYPE=auto` — L4·A100 은 bfloat16, T4 는 자동으로 float16 을 고릅니다
- `LLM_ENABLE_THINKING=false` — 켜두면 Gemma 4 가 사고 블록부터 써서 의도·카테고리 판정이 깨집니다

`MODEL_CACHE_DIR` 은 비워 두면 Colab 세션 로컬 디스크(`/root/.cache/huggingface`)를 씁니다.
**드라이브 용량이 부족하면 이렇게 두세요.** 대신 런타임이 끊기면 다음에 다시 켤 때
모델(약 18GB)을 처음부터 다시 받습니다. 드라이브 여유가 있다면 아래 주석 처리된
`/content/drive/MyDrive/hf_cache` 로 바꾸면 런타임이 끊겨도 다시 받지 않습니다.


In [ ]:
%%writefile .env
DEBUG=true
APP_ENV=local

DEVICE=auto
# 비워두면 세션 로컬 디스크 사용 (드라이브 용량 절약, 런타임 끊기면 재다운로드)
# 드라이브에 캐시해 두려면 아래 줄 주석을 풀고 위 줄을 지우세요.
MODEL_CACHE_DIR=
# MODEL_CACHE_DIR=/content/drive/MyDrive/hf_cache
PRELOAD_MODELS=false

EMBED_MODEL=BAAI/bge-m3

LLM_MODEL=google/gemma-4-E4B-it
LLM_LOAD_4BIT=true
LLM_4BIT_COMPUTE_DTYPE=auto
LLM_ENABLE_THINKING=false

CANDIDATE_TOP_K=3
CANDIDATE_MIN_SCORE=0.6
CASE_BLEND_WEIGHT=0.2
CANDIDATE_SCORING=multi
CASE_MODE=union
CASE_TOP_K=5
CASE_MIN_SCORE=0.5
GATE_ENABLED=true
FAQ_ENABLED=true
FAQ_TOP_K=3
FAQ_MIN_SCORE=0.6
CHAT_AGENT_ENABLED=true
CHAT_DEMO_ENABLED=true


## 6-A. 터널 없이 바로 확인 (가장 빠름)

Swagger를 띄우지 않고 파이프라인 함수를 직접 부릅니다. **여기까지만 해도 ②~⑥ 확인은 끝납니다.**

첫 실행은 가중치(bge-m3 2.2GB + Gemma 4 E4B 16GB)를 받느라 **5~15분** 걸립니다. 두 번째부터는 빠릅니다.

`pipeline.run(문장, user_id="...")` 의 `user_id` 는 "로그인한 사용자"로 취급할 이름입니다.
같은 `user_id` 로 등록한 민원만 조회·수정·취소할 수 있습니다. (비우면 서버 기본값 하나로 통일)


In [ ]:
from app.services import pipeline

민원 = '우리 동네 가로등이 꺼졌어요. 행복로 23길 버스정류장 앞 가로등 두 개가 일주일째 들어오지 않습니다. 밤에 골목이 너무 어두워서 위험합니다.'

결과 = pipeline.run(민원, user_id='colab-tester')

print(pipeline.render_result_text(결과))
print()
print('=' * 70)
print(pipeline.render_debug_text(결과))


### ⑥ 민원 DB 실제 동작 확인 - 접수 → 조회 → 수정 → 취소 (번호 없이)

앞서 KV 캐시·벡터DB 로 Gemma 가 판정한 결과가, 이제는 실제로 SQLite 민원 DB 에
쓰이고 읽힙니다. `pipeline.run()` 은 도구 호출 JSON 까지만 만드는 게 아니라
`tool_executor.py` 로 실제 INSERT/SELECT/UPDATE 를 실행한 결과(`결과.tool_result`)까지 돌려줍니다.

시민은 민원 번호를 기억하지 못하는 경우가 대부분이라, 조회·수정·삭제는 **"방금 넣은 포트홀 민원"처럼
주제·시점으로 가리켜도** 찾습니다. (도구 인자 `keyword` · `period` · `category`, 번호는 `complaint_id`)
- 찾은 민원이 **1건**이면 바로 실행합니다. (별도 확인 절차 없음)
- **여러 건**이거나 **아무 조건도 없으면**(예: "민원 취소할게요") 실행하지 않고 후보 목록을 돌려줍니다.
  `tool_result.needs_selection=True` 이고 `data` 에 후보가 들어 있으니, 화면에서 사용자가 고르게 하면 됩니다.
- `result_text` 의 `찾은 조건` 줄에서 어떤 조건으로 몇 건을 찾았는지 볼 수 있습니다.

In [ ]:
from app.services import pipeline

USER = 'colab-tester'

def 보기(결과):
    print(pipeline.render_result_text(결과))
    print('-' * 70)

# 1) 접수 두 건 - 바로 등록됩니다
r1 = pipeline.run('아파트 앞 도로에 포트홀이 생겨서 차가 덜컹거립니다. 보수해 주세요.', user_id=USER)
보기(r1)
r1b = pipeline.run('행복로 23길 가로등이 며칠째 꺼져 있어요.', user_id=USER)
보기(r1b)
print('=' * 70)

# 2) 조회 - 목록 / 주제로 한 건
보기(pipeline.run('제가 넣은 민원 목록 보여주세요.', user_id=USER))
보기(pipeline.run('방금 넣은 포트홀 민원 처리 상태 알려주세요.', user_id=USER))
print('=' * 70)

# 3) 수정 - 주제로 대상을 찾아 바로 반영 (번호 없음)
보기(pipeline.run('가로등 민원 위치를 행복로 23길에서 행복로 25길로 바꿔주세요.', user_id=USER))
print('=' * 70)

# 4) 여러 건이면 실행하지 않고 후보를 돌려줌 (needs_selection=True)
r4 = pipeline.run('오늘 넣은 민원 취소해 주세요.', user_id=USER)
보기(r4)
print('선택 필요:', r4.tool_result.needs_selection if r4.tool_result else None)
print('=' * 70)

# 5) 삭제(취소) - 주제로 한 건을 찾으면 바로 반영
보기(pipeline.run('포트홀 민원은 해결돼서 취소하고 싶어요.', user_id=USER))

# 6) 이력 - 접수/수정/취소가 순서대로 쌓였는지
보기(pipeline.run('포트홀 신고 처리 이력 보여줘.', user_id=USER))

### 의도별로 한 번씩 + 게이트 반려 확인

한 요청에 의도는 하나만 판정됩니다. 다섯 갈래가 제대로 갈리는지,
그리고 민원이 아닌 입력이 **⑤ 의도 판정에서 '해당없음'(6번)으로 반려**되는지 확인합니다.

**`의도 확신` 값을 눈여겨보세요.** 반려는 ④ 의 카테고리 유사도가 아니라 ⑤ 가 고른 의도로만 정해집니다.
④ 의 `1위 유사도`는 참고용이며, `CANDIDATE_MIN_SCORE` 미만이면 `low_confidence=True` 로 표시만 됩니다.
정상 민원이 반려되거나 잡담이 통과하면 `.env` 의 `GATE_ENABLED=false`(관찰 모드)로 두고
여러 문장을 돌려 `debug_text` 의 게이트 구간을 비교해 보세요.

이 셀의 조회·수정·삭제 예문은 주제·시점으로 민원을 가리킵니다. 조건에 맞는 민원이 없거나 여러 건이면
⑥ 결과가 '없음' 또는 '선택 필요'로 나올 수 있습니다. **의도·카테고리 판정이 맞는지 확인하는 셀**이고,
실제 등록→수정→취소 흐름은 바로 위 '⑥ 민원 DB 실제 동작 확인' 셀에서 봅니다.
조회·수정·삭제에서 주제가 드러나지 않으면(예: "어제 넣은 거") 카테고리는 `없음` 으로 판정됩니다.


In [ ]:
샘플 = [
    ('문의', '민원 처리 기간이 보통 얼마나 걸리나요?'),
    ('접수', '아파트 앞 도로에 포트홀이 생겨서 차가 덜컹거립니다. 보수해 주세요.'),
    ('조회', '제가 지난주에 넣은 민원 처리 상태가 어떻게 되는지 알려주세요.'),
    ('수정', '어제 접수한 민원 위치를 행복로 23길에서 15길로 바꿔주세요.'),
    ('삭제', '전에 넣은 소음 민원은 해결돼서 취소하고 싶어요.'),
    # ↓ ⑤ 의도 판정에서 '해당없음'으로 반려돼야 하는 입력
    ('반려', '오늘 점심 뭐 먹지'),
    ('반려', 'ㅁㄴㅇㄹ ㅁㄴㅇㄹ'),
    ('반려', '최신 스마트폰 특가 할인 이벤트 바로가기'),
]

for 기대, 문장 in 샘플:
    r = pipeline.run(문장, user_id='colab-tester')
    top1 = r.candidate.best
    유사도 = f'{top1.score:.3f}' if top1 else '-'
    참고 = ' (low)' if r.candidate.low_confidence else ''
    if r.rejected:
        판정 = '반려'
        상세 = f'의도 확신={r.gate.intent_score:.2f} (해당없음) | 1위 유사도={유사도}{참고}'
    else:
        판정 = r.llm.intent.name
        카테고리 = (f'{r.llm.category_name}({r.llm.category_choice.score:.2f})' if r.llm.category_name
                    else f'없음({r.llm.category_choice.score:.2f})' if r.llm.category_choice.scores else '-')
        상세 = (f'의도 확신={r.llm.intent_choice.score:.2f} | 카테고리={카테고리} '
                f'| 1위 유사도={유사도}{참고}')
        if r.tool_result is not None:
            상태 = '선택 필요' if r.tool_result.needs_selection else ('성공' if r.tool_result.ok else '오류')
            상세 += f' | ⑥ {상태}: {r.tool_result.message or r.tool_result.error}'
    표시 = 'OK  ' if 판정 == 기대 else '다름 '
    print(f'{표시} 기대={기대} 판정={판정} {상세} | {문장[:28]}')


### 대화 이어가기 확인 - "어느 민원인가요?" → "두 번째 거요" (Qwen-Agent)

수정·취소 대상이 여러 건이면 서버가 후보에 번호를 붙여 되묻고, 다음 말을 **Qwen-Agent**(함수 호출 프레임워크, 모델은 Gemma 공유)가 해석해
원래 요청을 고른 민원에 실행합니다. 같은 `session_id` 로 이어서 보내는 것이 핵심입니다. (README 5.12)

**📋 실행 결과**
- 첫 줄: `selection_asked / awaiting_selection` 과 후보 목록 (가로등 민원 2건 이상 접수돼 있어야 함 - 위 ⑥ 셀을 먼저 실행)
- 둘째 줄: `selected / idle`, `agent: 에이전트: select_complaint {'position': 2}` 와 `취소 완료 - …`
- `agent_available: False` 면 qwen-agent 가 설치되지 않은 것 - 2 단계를 다시 실행하세요. (그동안은 규칙으로 해석: `rules: …`)


In [ ]:
from app.services import chat

USER = 'colab-tester'
pipeline.run('행복로 23길 가로등이 꺼졌어요.', user_id=USER)        # 후보가 여러 건이 되도록 가로등 민원을 하나 더 접수
pipeline.run('중앙공원 입구 가로등이 깜빡거려요.', user_id=USER)
print(chat.status())

def 말하기(문장, sid=None):
    t = chat.handle(문장, user_id=USER, session_id=sid)
    how = f'  <- {t.resolution.method}: {t.resolution.note}' if t.resolution else ''
    print(f'[{t.kind} / {t.state}]{how}\n{t.reply}\n')
    return t

t = 말하기('가로등 민원 취소해 주세요')
if t.state == 'awaiting_selection':
    말하기('두 번째 거요', t.session_id)


## 6-B. Swagger 열기 (cloudflared 터널)

가입도 토큰도 필요 없습니다. 출력에 뜨는 주소 뒤에 `/docs`를 붙이면 Swagger가 열립니다.

> 터널 주소는 런타임마다 새로 생깁니다. 노트북을 닫으면 주소도 사라집니다.


대화 이어가기 데모 화면: 터널 주소 뒤에 `/chat-demo/demo.html` 을 붙여 여세요. (`CHAT_DEMO_ENABLED=true`)


In [ ]:
!pip install -q nest_asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared 준비 완료')


In [ ]:
import threading, time, nest_asyncio, uvicorn
from IPython import get_ipython

nest_asyncio.apply()

# 서버를 백그라운드 스레드로 띄웁니다
threading.Thread(
    target=lambda: uvicorn.run('app.main:app', host='0.0.0.0', port=8000, log_level='info'),
    daemon=True,
).start()
time.sleep(8)

# 터널 열기
get_ipython().system_raw('cloudflared tunnel --url http://127.0.0.1:8000 > /content/tunnel.log 2>&1 &')
time.sleep(10)

import re
로그 = open('/content/tunnel.log').read()
주소 = re.findall(r'https://[-a-z0-9]+\.trycloudflare\.com', 로그)
if 주소:
    print('Swagger 주소 :', 주소[0] + '/docs')
    print('상태 확인    :', 주소[0] + '/api/v1/analyze/status')
else:
    print('터널 주소를 못 찾았습니다. 아래 로그를 확인하세요.')
    print(로그[-2000:])


### 모델이 올라갔는지 확인

`loaded`가 `true`가 되면 다음 요청부터 빠릅니다. `runtime`에서 4bit + bfloat16(T4 는 float16)으로 올라갔는지 볼 수 있습니다.


In [ ]:
import requests, json
r = requests.get('http://127.0.0.1:8000/api/v1/analyze/status').json()
print(json.dumps(r, ensure_ascii=False, indent=2))


In [ ]:
# Swagger 대신 여기서 바로 쏴볼 수도 있습니다
import requests, json
r = requests.post(
    'http://127.0.0.1:8000/api/v1/analyze/text',
    json={'text': '우리 동네 가로등이 꺼졌어요'},
    timeout=1800,
).json()
print(r['result_text'])
print()
print(r['debug']['debug_text'])


## 7. (선택) 벡터DB 파일을 PC 로 가져가기

PC 의 PostgreSQL(pgvector)에 사례를 넣으려면 여기서 만든 벡터 파일 두 개가 필요합니다. (README 1-6 의 5단계)
PC 에는 bge-m3 가 없어 직접 임베딩할 수 없기 때문입니다.

`MyDrive/backend/data/` 에 이미 있으니 드라이브에서 바로 받아도 됩니다.
PC 의 `backend\data\` 에 복사한 뒤 `python scripts/sync_cases.py` 를 실행하세요.

In [ ]:
from app.services import case_store
case_store.load_or_build()          # 6-A 를 이미 실행했다면 캐시만 확인합니다
print(case_store.status())

from google.colab import files
files.download('data/cases.npy')
files.download('data/cases.meta.json')

## 문제가 생기면

| 증상 | 원인과 해결 |
|---|---|
| 첫 요청이 5~15분 | 정상입니다. 가중치 약 18GB를 받는 중입니다. `MODEL_CACHE_DIR`을 드라이브로 두면 다음부터 건너뜁니다. |
| `KeyError: 'gemma4'` | transformers가 5.5 미만입니다. 2번 셀 설치 후 **3번 셀로 재시작**했는지 확인하세요. |
| 판정 확률이 전부 비슷하거나 이상함 (T4) | T4 는 float16 이라 Gemma 값이 넘칠 수 있습니다. L4 로 바꾸세요. |
| `CUDA out of memory` | `.env`의 `LLM_LOAD_4BIT=true`를 확인하세요. 그래도 나면 런타임을 재시작하고 다시 올리세요. |
| 응답에 사고 내용이 섞임 | `.env`의 `LLM_ENABLE_THINKING=false`를 확인하세요. |
| bf16 관련 경고 | `LLM_4BIT_COMPUTE_DTYPE=auto`로 두세요. GPU 에 맞게 자동으로 고릅니다. |
| 이미지 업로드가 503 | Colab에는 PaddleOCR을 안 깔았습니다. 정상이며, 추출은 윈도우 쪽에서 하세요. |
| 90분 방치 후 끊김 | Colab 무료 티어 제한입니다. 런타임이 끊기면 4번 셀부터 다시 실행하세요. |
